# Penyelarasan Warna dan Perbaikan Citra Malam Hari — Tahap 2: metode klasik

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prodi-TI-IBIKesatuan/pcd2026-ti23ka-t2/blob/main/notebooks/02_metode_klasik.ipynb)

Kelompok ini wajib **3 metode klasik**. Pilihan yang disarankan:

- Koreksi gamma
- CLAHE pada kanal L (LAB) atau V (HSV)
- Retinex (single-scale atau multi-scale)
- White balance gray-world

Setiap metode adalah satu fungsi. Evaluasi memakai `pcd_proyek.evaluasi`, sehingga metrik sama dengan kelompok lain.

In [1]:
# Siapkan lingkungan: salin repo ini dan hubungkan Google Drive kelompok.
import pathlib
import subprocess
import sys

import cv2
import numpy as np

REPO = "Prodi-TI-IBIKesatuan/pcd2026-ti23ka-t2"
try:
    import google.colab  # noqa: F401
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    AKAR = pathlib.Path("/content/repo")
    if AKAR.exists():
        subprocess.run(["git", "-C", str(AKAR), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/{REPO}.git", str(AKAR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    FOLDER_KERJA = pathlib.Path("/content/drive/MyDrive/pcd2026-ti23ka-t2")
    if not FOLDER_KERJA.exists():
        raise FileNotFoundError(f"{FOLDER_KERJA} tidak ada. Tambahkan pintasan folder Drive kelompok ke Drive Saya (lihat PANDUAN.md).")
else:  # dijalankan di komputer sendiri dari folder notebooks/
    AKAR = pathlib.Path.cwd().resolve().parent
    FOLDER_KERJA = AKAR / "data"
sys.path.insert(0, str(AKAR))

from pcd_proyek import evaluasi as ev, metrik, util  # noqa: E402

FOLDER_CITRA = FOLDER_KERJA / "citra" if (FOLDER_KERJA / "citra").exists() else AKAR / "data" / "sample"
BERKAS_ANOTASI = FOLDER_KERJA / "anotasi.csv"
FOLDER_FIGUR = FOLDER_KERJA / "figures"
BERKAS_HASIL = FOLDER_KERJA / "hasil_metode.json"
KOLOM_ANOTASI = ['berkas_gelap', 'berkas_referensi', 'sumber']
JUMLAH_ANGGOTA = 5
print("Citra   :", FOLDER_CITRA)
print("Anotasi :", BERKAS_ANOTASI, "(ada)" if BERKAS_ANOTASI.exists() else "(belum ada)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Citra   : /content/drive/MyDrive/pcd2026-ti23ka-t2/citra
Anotasi : /content/drive/MyDrive/pcd2026-ti23ka-t2/anotasi.csv (ada)


In [2]:
anotasi = util.baca_anotasi(BERKAS_ANOTASI, KOLOM_ANOTASI)
UJI_CEPAT = None  # isi angka (mis. 20) saat mencoba-coba agar cepat; kembalikan ke None untuk hasil akhir
print(len(anotasi), "baris anotasi")

330 baris anotasi


## Metode klasik 1

**Nama metode dan alasannya:** Koreksi Gamma. Metode ini dipilih karena citra malam hari memiliki distribusi intensitas piksel yang menumpuk di area gelap.

In [3]:
def metode_klasik_1(citra_bgr):
    gamma = 0.5
    tabel_lookup = np.array(
        [((i / 255.0) ** gamma) * 255 for i in range(256)],
        dtype=np.uint8
    )
    citra_hasil_bgr = cv2.LUT(citra_bgr, tabel_lookup)
    return citra_hasil_bgr

In [4]:
NAMA_1 = "Koreksi Gamma (gamma=0.5)"

## Metode klasik 2

**Nama metode dan alasannya:** CLAHE pada kanal Lightness di ruang warna CIE LAB. Dengan memisahkan informasi pencahayaan kanal Lightness dari informasi warna, peningkatan kontras lokal dapat dilakukan secara adaptif di tiap blok kecil citra tanpa merusak keseimbangan warna asli

In [5]:
def metode_klasik_2(citra_bgr):
    citra_lab = cv2.cvtColor(citra_bgr, cv2.COLOR_BGR2LAB)
    kanal_l, kanal_a, kanal_b = cv2.split(citra_lab)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    kanal_l_clahe = clahe.apply(kanal_l)

    citra_lab_hasil = cv2.merge((kanal_l_clahe, kanal_a, kanal_b))
    citra_hasil_bgr = cv2.cvtColor(citra_lab_hasil, cv2.COLOR_LAB2BGR)
    return citra_hasil_bgr

In [6]:
NAMA_2 = "CLAHE Kanal L (LAB)"

## Metode klasik 3

**Nama metode dan alasannya:** Kombinasi White Balance Gray-World dan Multi-Scale Retinex. Citra malam hari sering mengalami pergeseran warna kekuningan atau kebiruan akibat pencahayaan buatan serta rentang dinamis yang sempit. *Gray-World* menyelaraskan rata-rata kanal warna agar tampak natural, sedangkan MSR mengestimasi komponen iluminasi.

In [7]:
def metode_klasik_3(citra_bgr):
    """Metode klasik 3: Penyelarasan warna Gray-World dilanjutkan Multi-Scale Retinex."""
    # 1. White Balance Gray-World
    citra_float = citra_bgr.astype(np.float32) + 1.0
    rata_b, rata_g, rata_r = np.mean(citra_float, axis=(0, 1))
    rata_abu = (rata_b + rata_g + rata_r) / 3.0

    citra_wb = np.empty_like(citra_float)
    citra_wb[:, :, 0] = citra_float[:, :, 0] * (rata_abu / rata_b)
    citra_wb[:, :, 1] = citra_float[:, :, 1] * (rata_abu / rata_g)
    citra_wb[:, :, 2] = citra_float[:, :, 2] * (rata_abu / rata_r)
    citra_wb = np.clip(citra_wb, 1.0, 255.0)

    # 2. Multi-Scale Retinex (MSR)
    skala_sigma = [15, 80, 250]
    retinex = np.zeros_like(citra_wb)
    for sigma in skala_sigma:
        blur = cv2.GaussianBlur(citra_wb, (0, 0), sigma)
        retinex += np.log10(citra_wb) - np.log10(np.maximum(blur, 1.0))
    retinex /= len(skala_sigma)

    citra_hasil = np.zeros_like(retinex)
    for c in range(3):
        bawah, atas = np.percentile(retinex[:, :, c], (1, 99))
        kanal = (retinex[:, :, c] - bawah) / (atas - bawah + 1e-6) * 255.0
        citra_hasil[:, :, c] = np.clip(kanal, 0, 255)

    return citra_hasil.astype(np.uint8)

In [8]:
NAMA_3 = "Gray-World + Multi-Scale Retinex"

## Simpan ke GitHub

1. **Runtime → Restart session and run all**, pastikan tidak ada error.
2. **File → Save a copy in GitHub** → repo `Prodi-TI-IBIKesatuan/pcd2026-ti23ka-t2` → **branch Anda** (bukan `main`) → path tetap `notebooks/02_metode_klasik.ipynb`.
3. Buka *pull request* ke `main` di github.com. Langkah lengkap: `PANDUAN.md`.